# Test Case Overview

8-panel overview of the cases selected for hyperparameter analysis (031).
Each row shows one case with monthly-aggregated forcing fields and the best baseline route.

| Case | Month | Direction | Speed | Regime |
|------|-------|-----------|-------|--------|
| Aug E 8kn | Aug | eastward | 8 kn | current-dominated |
| Aug E 12kn | Aug | eastward | 12 kn | current, reduced |
| Aug W 8kn | Aug | westward | 8 kn | calm, slow |
| Aug W 12kn | Aug | westward | 12 kn | calm, fast |
| Jan E 8kn | Jan | eastward | 8 kn | wave + current |
| Jan E 12kn | Jan | eastward | 12 kn | wave-dominated |
| Jan W 8kn | Jan | westward | 8 kn | wave detour, slow |
| Jan W 12kn | Jan | westward | 12 kn | wave detour, fast |

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cmocean
import warnings

from experiment_params import FORCING_SCENARIOS
from ship_routing.core.data import load_currents, load_waves, load_winds
from load_tuning_results import add_derived_features, filter_suspicious_routes

warnings.filterwarnings("ignore")

In [2]:
CASES = [
    {"month": 8, "direction": "eastward", "speed": 8, "journey": "Atlantic_forward",
     "label": "Aug E 8kn"},
    {"month": 8, "direction": "eastward", "speed": 12, "journey": "Atlantic_forward",
     "label": "Aug E 12kn"},
    {"month": 8, "direction": "westward", "speed": 8, "journey": "Atlantic_backward",
     "label": "Aug W 8kn"},
    {"month": 8, "direction": "westward", "speed": 12, "journey": "Atlantic_backward",
     "label": "Aug W 12kn"},
    {"month": 1, "direction": "eastward", "speed": 8, "journey": "Atlantic_forward",
     "label": "Jan E 8kn"},
    {"month": 1, "direction": "eastward", "speed": 12, "journey": "Atlantic_forward",
     "label": "Jan E 12kn"},
    {"month": 1, "direction": "westward", "speed": 8, "journey": "Atlantic_backward",
     "label": "Jan W 8kn"},
    {"month": 1, "direction": "westward", "speed": 12, "journey": "Atlantic_backward",
     "label": "Jan W 12kn"},
]

# Map extent (Atlantic crossing with buffer)
LON_MIN, LON_MAX = -85, -5
LAT_MIN, LAT_MAX = 20, 60

# Projection center
LON_CENT, LAT_CENT = -45.75, 40.0

FIG_DIR = "../figures"

## Load forcing data

In [3]:
baseline = FORCING_SCENARIOS["baseline"]
data_prefix = Path("..")

ds_currents = load_currents(data_prefix / baseline["currents_path"])
ds_currents = ds_currents.assign(
    speed=(ds_currents.to_array() ** 2).sum("variable") ** 0.5
)
print(f"Currents: {dict(ds_currents.dims)}")

ds_waves = load_waves(data_prefix / baseline["waves_path"])
print(f"Waves: {dict(ds_waves.dims)}")

ds_winds = load_winds(data_prefix / baseline["winds_path"])
ds_winds = ds_winds.resample(time="1D").mean()
ds_winds = ds_winds.assign(
    speed=(ds_winds.to_array() ** 2).sum("variable") ** 0.5
)
print(f"Winds: {dict(ds_winds.dims)}")

Currents: {'time': 365, 'lat': 661, 'lon': 1321}
Waves: {'time': 2920, 'lat': 276, 'lon': 551}
Winds: {'time': 365, 'lat': 440, 'lon': 880}


In [ ]:
monthly_current_speed = ds_currents.speed.resample(time="1MS").mean()
monthly_wave_height = ds_waves.wh.resample(time="1MS").quantile(0.9)
monthly_wind_speed = ds_winds.speed.resample(time="1MS").quantile(0.9)

# Color ranges from q98
current_vmax = float(monthly_current_speed.quantile(0.98))
wave_vmax = float(monthly_wave_height.quantile(0.98))
wind_vmax = float(monthly_wind_speed.quantile(0.98))

print(f"Current speed range: 0 - {current_vmax:.2f} m/s")
print(f"Wave height q90 range: 0 - {wave_vmax:.2f} m")
print(f"Wind speed q90 range: 0 - {wind_vmax:.2f} m/s")

## Load best routes per case

In [ ]:
gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)

# Baseline, no hazard penalty, best elite
gdf = gdf[
    (gdf.forcing_scenario_name == "baseline")
    & (gdf.hyper_hazard_penalty_multiplier == 0.0)
    & (gdf.n_elite == 0)
].copy()

gdf["month_num"] = pd.to_datetime(gdf.journey_time_start.astype(str)).dt.month
gdf["direction"] = gdf.journey_name.map(
    {"Atlantic_forward": "eastward", "Atlantic_backward": "westward"}
)

# Pick best route (lowest cost) per case
best_routes = {}
for case in CASES:
    mask = (
        (gdf.month_num == case["month"])
        & (gdf.direction == case["direction"])
        & (gdf.journey_speed_knots == case["speed"])
    )
    sub = gdf[mask].sort_values("elite_cost_absolute")
    best_routes[case["label"]] = sub.iloc[0]
    cost_TJ = sub.iloc[0].elite_cost_absolute / 1e12
    print(f"{case['label']}: best cost = {cost_TJ:.3f} TJ (from {mask.sum()} configs)")

0.57% suspicious routes
Aug E 8kn: best cost = 1.924 TJ (from 248 configs)
Aug E 12kn: best cost = 4.828 TJ (from 291 configs)
Aug W 8kn: best cost = 2.870 TJ (from 252 configs)
Aug W 12kn: best cost = 4.964 TJ (from 309 configs)
Jan E 8kn: best cost = 7.939 TJ (from 212 configs)
Jan E 12kn: best cost = 11.501 TJ (from 304 configs)
Jan W 8kn: best cost = 6.326 TJ (from 261 configs)
Jan W 12kn: best cost = 9.141 TJ (from 307 configs)


## Overview plot

In [ ]:
proj = cartopy.crs.Stereographic(
    central_latitude=LAT_CENT, central_longitude=LON_CENT
)
transform = cartopy.crs.PlateCarree()

MONTHS = [
    {"month": 8, "label": "Aug 2021"},
    {"month": 1, "label": "Jan 2021"},
]
ROUTE_STYLES = [
    {"direction": "eastward", "speed": 8, "color": "magenta", "ls": "-", "label": "E 8kn"},
    {"direction": "eastward", "speed": 12, "color": "magenta", "ls": "--", "label": "E 12kn"},
    {"direction": "westward", "speed": 8, "color": "cyan", "ls": "-", "label": "W 8kn"},
    {"direction": "westward", "speed": 12, "color": "cyan", "ls": "--", "label": "W 12kn"},
]
COL_LABELS = ["current speed", "wave height q90", "wind speed q90"]

fig, axes = plt.subplots(
    len(MONTHS), 3,
    subplot_kw={"projection": proj},
    figsize=(3 * 4, len(MONTHS) * 3),
    sharex=True,
    sharey=True,
)
fig.set_dpi(300)

for i, minfo in enumerate(MONTHS):
    month_idx = minfo["month"] - 1

    # Current speed
    monthly_current_speed.isel(time=month_idx).plot(
        vmin=0, vmax=current_vmax, extend="max",
        cmap=cmocean.cm.speed, ax=axes[i, 0],
        transform=transform, rasterized=True, add_colorbar=False,
    )

    # Wave height q90
    monthly_wave_height.isel(time=month_idx).plot(
        vmin=0, vmax=wave_vmax, extend="max",
        cmap=cmocean.cm.amp, ax=axes[i, 1],
        transform=transform, rasterized=True, add_colorbar=False,
    )

    # Wind speed q90
    monthly_wind_speed.isel(time=month_idx).plot(
        vmin=0, vmax=wind_vmax, extend="max",
        cmap=cmocean.cm.speed, ax=axes[i, 2],
        transform=transform, rasterized=True, add_colorbar=False,
    )

    # Override xarray auto-titles: "Month — field"
    for j, col_label in enumerate(COL_LABELS):
        axes[i, j].set_title(f"{minfo['label']} — {col_label}")

    # Overlay all 4 routes for this month
    for rs in ROUTE_STYLES:
        # Route keys use short month names (Aug/Jan)
        month_short = minfo["label"].split()[0]
        key = f"{month_short} {rs['label']}"
        route_geom = best_routes[key].geometry
        for j in range(3):
            axes[i, j].plot(
                *route_geom.xy,
                transform=transform,
                color=rs["color"], linestyle=rs["ls"],
                linewidth=1.5, label=rs["label"],
            )

    # Map features
    for j in range(3):
        axes[i, j].coastlines()
        axes[i, j].gridlines(draw_labels=False)
        axes[i, j].set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])

# Legend (from first panel, deduplicated)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles[:4], labels[:4], loc="lower center", ncol=4, frameon=False)

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(f"{FIG_DIR}/030_test_case_overview.png", dpi=150)
fig.savefig(f"{FIG_DIR}/030_test_case_overview.pdf", dpi=150)